In [1]:
import sys

print(sys.executable)

/Users/binguyen/.venvs/lede/bin/python


In [145]:
import os
import geopandas as gpd
import pandas as pd
import pydash

In [2]:
script_dir = os.path.dirname(os.path.abspath('crosswalk.ipynb'))

In [3]:
shp_file_path = os.path.join(script_dir, '..', 'data', 'raw', 'geography', 'SBE_PRECINCTS_20251212', 'SBE_PRECINCTS_20251212.shp')

In [6]:
precincts_gdf = gpd.read_file(shp_file_path)

In [7]:
precincts_gdf["county_key"] = (
    precincts_gdf["county_nam"]
    .astype(str)
    .str.strip()
    .str.upper()
)

precincts_gdf["precinct_raw"] = (
    precincts_gdf["prec_id"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [8]:
contest_summary_file_path = os.path.join(script_dir, '..', 'data', 'processed', 'us_senate_rep_contest_summary.csv')

In [10]:
contest_summary_df = pd.read_csv(contest_summary_file_path)

In [13]:
contest_summary_df["county_key"] = (
    contest_summary_df["County"]
    .astype(str)
    .str.strip()
    .str.upper()
)

contest_summary_df["precinct_raw"] = (
    contest_summary_df["Precinct"]
    .astype(str)
    .str.strip()
    .str.upper()
)

## Check that county + precinct is actually unique

In [14]:
geo_dupes = precincts_gdf[
    precincts_gdf.duplicated(
        ["county_key", "precinct_raw"],
        keep=False
    )
].sort_values(["county_key", "precinct_raw"])

In [16]:
summary_dupes = contest_summary_df[
    contest_summary_df.duplicated(
        ["county_key", "precinct_raw"],
        keep=False
    )
].sort_values(["county_key", "precinct_raw"])

In [18]:
print("Geo duplicates:", len(geo_dupes))
print("Summary duplicates:", len(summary_dupes))

Geo duplicates: 0
Summary duplicates: 0


## Do an outer merge for diagnosis

In [19]:
diagnostic = precincts_gdf[
    ["county_key", "precinct_raw"]
].merge(
    contest_summary_df[
        ["county_key", "precinct_raw"]
    ],
    on=["county_key", "precinct_raw"],
    how="outer",
    indicator=True
)

In [20]:
diagnostic["_merge"].value_counts()

_merge
both          2223
right_only     410
left_only      409
Name: count, dtype: int64

Those last two groups are what we need to investigate.

- both = exact matches
- left_only = polygon exists but no election-result match
- right_only = election-result precinct exists but no polygon match

## Pull out all mismatches

In [21]:
geo_unmatched = (
    diagnostic[
        diagnostic["_merge"] == "left_only"
    ]
    [["county_key", "precinct_raw"]]
    .sort_values(["county_key", "precinct_raw"])
)

results_unmatched = (
    diagnostic[
        diagnostic["_merge"] == "right_only"
    ]
    [["county_key", "precinct_raw"]]
    .sort_values(["county_key", "precinct_raw"])
)

In [22]:
print(geo_unmatched.to_string(index=False))

 county_key precinct_raw
   ALAMANCE           01
   ALAMANCE           02
   ALAMANCE          035
   ALAMANCE           04
   ALAMANCE           05
   ALAMANCE          063
   ALAMANCE          064
   ALAMANCE           07
  ALLEGHANY           01
  ALLEGHANY           04
       ASHE           02
       ASHE           03
       ASHE           04
       ASHE           06
       ASHE           07
       ASHE           08
       ASHE           09
      AVERY           01
      AVERY           02
      AVERY           03
      AVERY           04
      AVERY           05
      AVERY           06
      AVERY           07
      AVERY           08
      AVERY           09
   BUNCOMBE         01.1
   BUNCOMBE         02.1
   BUNCOMBE         03.1
   BUNCOMBE         04.1
   BUNCOMBE         05.1
   BUNCOMBE         06.1
   BUNCOMBE         07.1
   BUNCOMBE         08.2
   BUNCOMBE         08.3
   BUNCOMBE         09.1
      BURKE         0001
      BURKE         0003
      BURKE         0011


In [23]:
print(results_unmatched.to_string(index=False))

 county_key precinct_raw
   ALAMANCE            1
   ALAMANCE            2
   ALAMANCE           35
   ALAMANCE            4
   ALAMANCE            5
   ALAMANCE           63
   ALAMANCE           64
   ALAMANCE            7
  ALLEGHANY            1
  ALLEGHANY            4
       ASHE            2
       ASHE            3
       ASHE            4
       ASHE            6
       ASHE            7
       ASHE            8
       ASHE            9
      AVERY            1
      AVERY            2
      AVERY            3
      AVERY            4
      AVERY            5
      AVERY            6
      AVERY            7
      AVERY            8
      AVERY            9
   BUNCOMBE          1.1
   BUNCOMBE          2.1
   BUNCOMBE          3.1
   BUNCOMBE          4.1
   BUNCOMBE          5.1
   BUNCOMBE          6.1
   BUNCOMBE          7.1
   BUNCOMBE          8.2
   BUNCOMBE          8.3
   BUNCOMBE          9.1
      BURKE            1
      BURKE           11
      BURKE           12


## Compare mismatches within the same county

In [24]:
candidates = results_unmatched.merge(
    geo_unmatched,
    on="county_key",
    how="left",
    suffixes=("_results", "_geo")
)

In [25]:
candidates.head(30)

,county_key,precinct_raw_results,precinct_raw_geo
0,ALAMANCE,1,01
1,ALAMANCE,1,02
2,ALAMANCE,1,035
3,ALAMANCE,1,04
4,ALAMANCE,1,05
5,ALAMANCE,1,063
6,ALAMANCE,1,064
7,ALAMANCE,1,07
8,ALAMANCE,2,01
9,ALAMANCE,2,02


## Create a normalization field only for diagnosis

In [26]:
def numeric_normalized(value):
    value = str(value).strip().upper()

    if value.isdigit():
        return str(int(value))

    return value

In [27]:
geo_unmatched = geo_unmatched.copy()
results_unmatched = results_unmatched.copy()

geo_unmatched["numeric_key"] = (
    geo_unmatched["precinct_raw"]
    .apply(numeric_normalized)
)

results_unmatched["numeric_key"] = (
    results_unmatched["precinct_raw"]
    .apply(numeric_normalized)
)

In [28]:
geo_unmatched

,county_key,precinct_raw,numeric_key
0,ALAMANCE,01,1
1,ALAMANCE,02,2
2,ALAMANCE,035,35
9,ALAMANCE,04,4
10,ALAMANCE,05,5
...,...,...,...
2937,WAYNE,05,5
2938,WAYNE,06,6
2939,WAYNE,07,7
2940,WAYNE,08,8


In [29]:
results_unmatched

,county_key,precinct_raw,numeric_key
22,ALAMANCE,1,1
40,ALAMANCE,2,2
41,ALAMANCE,35,35
42,ALAMANCE,4,4
43,ALAMANCE,5,5
...,...,...,...
2965,WAYNE,5,5
2966,WAYNE,6,6
2967,WAYNE,7,7
2968,WAYNE,8,8


## Find possible leading-zero matches

In [30]:
zero_candidates = results_unmatched.merge(
    geo_unmatched,
    on=["county_key", "numeric_key"],
    how="inner",
    suffixes=("_results", "_geo")
)

In [33]:
zero_candidates[
    [
        "county_key",
        "precinct_raw_results",
        "precinct_raw_geo"
    ]
].sort_values(
    ["county_key", "precinct_raw_results"]
).tail(50)

,county_key,precinct_raw_results,precinct_raw_geo
348,UNION,32,032
349,UNION,33,033
350,UNION,34,034
351,UNION,35,035
352,UNION,36,036
353,UNION,39,039
354,UNION,4,004
355,UNION,40,040
356,UNION,41,041
357,UNION,42,042


## Detect ambiguous normalization
This is the crucial safeguard given what you discovered.

Suppose within one county the shapefile contains BOTH:
- 1
- 01

and the election results contain:
- 1

Then stripping leading zeros produces:
- 1  → 1
- 01 → 1

We have no basis for deciding which polygon is correct.

Find those cases:

Anything returned here should be manually investigated.

Do not automatically join it.

In [34]:
ambiguity = (
    zero_candidates
    .groupby(
        ["county_key", "precinct_raw_results"]
    )
    .size()
    .reset_index(name="possible_geo_matches")
)

ambiguity = ambiguity[
    ambiguity["possible_geo_matches"] > 1
]

In [35]:
ambiguity

,county_key,precinct_raw_results,possible_geo_matches


## Separate safe matches from ambiguous matches

In [36]:
match_counts = (
    zero_candidates
    .groupby(
        ["county_key", "precinct_raw_results"]
    )
    .size()
    .rename("match_count")
    .reset_index()
)

zero_candidates = zero_candidates.merge(
    match_counts,
    on=["county_key", "precinct_raw_results"]
)

In [37]:
safe_zero_matches = zero_candidates[
    zero_candidates["match_count"] == 1
].copy()

In [38]:
ambiguous_zero_matches = zero_candidates[
    zero_candidates["match_count"] > 1
].copy()

In [39]:
safe_zero_matches

,county_key,precinct_raw_results,numeric_key,precinct_raw_geo,match_count
0,ALAMANCE,1,1,01,1
1,ALAMANCE,2,2,02,1
2,ALAMANCE,35,35,035,1
3,ALAMANCE,4,4,04,1
4,ALAMANCE,5,5,05,1
...,...,...,...,...,...
393,WAYNE,5,5,05,1
394,WAYNE,6,6,06,1
395,WAYNE,7,7,07,1
396,WAYNE,8,8,08,1


In [40]:
ambiguous_zero_matches

,county_key,precinct_raw_results,numeric_key,precinct_raw_geo,match_count


## Build an explicit crosswalk

In [41]:
# For exact matches

exact_crosswalk = diagnostic[
    diagnostic["_merge"] == "both"
][
    ["county_key", "precinct_raw"]
].copy()

exact_crosswalk["results_id"] = exact_crosswalk["precinct_raw"]
exact_crosswalk["geo_id"] = exact_crosswalk["precinct_raw"]
exact_crosswalk["match_method"] = "exact"

exact_crosswalk = exact_crosswalk[
    ["county_key", "results_id", "geo_id", "match_method"]
]

In [42]:
# For safe zero matches

zero_crosswalk = safe_zero_matches[
    [
        "county_key",
        "precinct_raw_results",
        "precinct_raw_geo"
    ]
].copy()

zero_crosswalk = zero_crosswalk.rename(columns={
    "precinct_raw_results": "results_id",
    "precinct_raw_geo": "geo_id"
})

zero_crosswalk["match_method"] = "leading_zero"

In [43]:
crosswalk = pd.concat(
    [
        exact_crosswalk,
        zero_crosswalk
    ],
    ignore_index=True
)

In [47]:
crosswalk["match_method"].value_counts()

match_method
exact           2223
leading_zero     398
Name: count, dtype: int64

## Compare crosswalk against precincts source data

In [48]:
print("Total precinct polygons:", len(precincts_gdf))

print(
    "Unique county + precinct:",
    precincts_gdf[
        ["county_key", "precinct_raw"]
    ].drop_duplicates().shape[0]
)

print(
    "Counties:",
    precincts_gdf["county_key"].nunique()
)

Total precinct polygons: 2632
Unique county + precinct: 2632
Counties: 100


## Determine which polygons have a Senate result

In [49]:
senate_geo_keys = crosswalk[
    ["county_key", "geo_id"]
].drop_duplicates()

In [50]:
senate_geo_keys = senate_geo_keys.rename(
    columns={"geo_id": "precinct_raw"}
)

In [51]:
geo_contest_check = precincts_gdf.merge(
    senate_geo_keys,
    on=["county_key", "precinct_raw"],
    how="left",
    indicator=True
)

In [52]:
geo_contest_check["_merge"].value_counts()

# Because this is a left join starting from geometry:
# both = polygon has a Senate Republican contest record
# left_only = polygon exists but has no Senate Republican record

_merge
both          2621
left_only       11
right_only       0
Name: count, dtype: int64

## Look at the polygons with no Senate record

In [53]:
no_senate_record = (
    geo_contest_check[
        geo_contest_check["_merge"] == "left_only"
    ]
    [
        ["county_key", "precinct_raw"]
    ]
    .sort_values(
        ["county_key", "precinct_raw"]
    )
)

In [55]:
print(
    "Polygons without Senate record:",
    len(no_senate_record)
)

Polygons without Senate record: 11


In [54]:
print(no_senate_record.to_string(index=False))

 county_key precinct_raw
   BUNCOMBE         01.1
   BUNCOMBE         02.1
   BUNCOMBE         03.1
   BUNCOMBE         04.1
   BUNCOMBE         05.1
   BUNCOMBE         06.1
   BUNCOMBE         07.1
   BUNCOMBE         08.2
   BUNCOMBE         08.3
   BUNCOMBE         09.1
MECKLENBURG        078.1


In [58]:
contest_summary_df["contest_votes"].eq(0).value_counts()

contest_votes
False    2617
True       16
Name: count, dtype: int64

In [60]:
summary_crosswalk_check = contest_summary_df.merge(
    crosswalk[
        ["county_key", "results_id"]
    ],
    left_on=[
        "county_key",
        "precinct_raw"
    ],
    right_on=[
        "county_key",
        "results_id"
    ],
    how="left",
    indicator=True
)

unmatched_results = (
    summary_crosswalk_check[
        summary_crosswalk_check["_merge"] == "left_only"
    ]
    .copy()
)

print(
    unmatched_results[
        [
            "county_key",
            "precinct_raw",
            "contest_votes"
        ]
    ]
    .sort_values(
        ["county_key", "precinct_raw"]
    )
    .to_string(index=False)
)

 county_key precinct_raw  contest_votes
   BUNCOMBE          1.1             29
   BUNCOMBE          2.1             19
   BUNCOMBE          3.1             22
   BUNCOMBE          4.1             37
   BUNCOMBE          5.1            113
   BUNCOMBE          6.1             68
   BUNCOMBE          7.1             35
   BUNCOMBE          8.2             53
   BUNCOMBE          8.3             70
   BUNCOMBE          9.1             76
  HENDERSON           CV            115
MECKLENBURG         78.1             36


In [61]:
print("Unmatched result precincts:", len(unmatched_results))

Unmatched result precincts: 12


In [63]:
print(
    unmatched_results[
        [
            "county_key",
            "precinct_raw",
            "contest_votes"
        ]
    ]
    .sort_values(
        ["contest_votes", "county_key", "precinct_raw"]
    )
    .to_string(index=False)
)

 county_key precinct_raw  contest_votes
   BUNCOMBE          2.1             19
   BUNCOMBE          3.1             22
   BUNCOMBE          1.1             29
   BUNCOMBE          7.1             35
MECKLENBURG         78.1             36
   BUNCOMBE          4.1             37
   BUNCOMBE          8.2             53
   BUNCOMBE          6.1             68
   BUNCOMBE          8.3             70
   BUNCOMBE          9.1             76
   BUNCOMBE          5.1            113
  HENDERSON           CV            115


In [64]:
print(
    unmatched_results["county_key"]
    .value_counts()
    .sort_index()
)

county_key
BUNCOMBE       10
HENDERSON       1
MECKLENBURG     1
Name: count, dtype: int64


In [65]:
print(
    no_senate_record["county_key"]
    .value_counts()
    .sort_index()
)

county_key
BUNCOMBE       10
MECKLENBURG     1
Name: count, dtype: int64


In [66]:
for county in ["BUNCOMBE", "MECKLENBURG"]:

    print(f"\n--- {county} ---")

    print("\nRESULTS:")
    print(
        unmatched_results.loc[
            unmatched_results["county_key"] == county,
            ["precinct_raw", "contest_votes"]
        ]
        .sort_values("precinct_raw")
        .to_string(index=False)
    )

    print("\nGEOGRAPHY:")
    print(
        no_senate_record.loc[
            no_senate_record["county_key"] == county,
            ["precinct_raw"]
        ]
        .sort_values("precinct_raw")
        .to_string(index=False)
    )


--- BUNCOMBE ---

RESULTS:
precinct_raw  contest_votes
         1.1             29
         2.1             19
         3.1             22
         4.1             37
         5.1            113
         6.1             68
         7.1             35
         8.2             53
         8.3             70
         9.1             76

GEOGRAPHY:
precinct_raw
        01.1
        02.1
        03.1
        04.1
        05.1
        06.1
        07.1
        08.2
        08.3
        09.1

--- MECKLENBURG ---

RESULTS:
precinct_raw  contest_votes
        78.1             36

GEOGRAPHY:
precinct_raw
       078.1


In [67]:
henderson_unmatched = unmatched_results[
    unmatched_results["county_key"] == "HENDERSON"
]

print(
    henderson_unmatched[
        [
            "precinct_raw",
            "contest_votes"
        ]
    ].to_string(index=False)
)

precinct_raw  contest_votes
          CV            115


In [70]:
print("RESULT PRECINCTS")

print(
    contest_summary_df.loc[
        contest_summary_df["county_key"] == "HENDERSON",
        ["precinct_raw", "contest_votes"]
    ]
    .sort_values("precinct_raw")
    .to_string(index=False)
)

RESULT PRECINCTS
precinct_raw  contest_votes
          AR            209
          AT            530
          BC            142
          BK            104
          CB            236
          CC            466
          CV            115
          ED            473
          EF            324
          ES            297
          EV            401
          FL            376
          FR            622
          GM            143
          GR            477
          HC            360
          HS            399
        HV-1            271
        HV-2            169
        HV-3            245
          LJ            296
          LP            271
          MG            136
          NB            528
          NE            216
          NM            366
          NW            216
          PR            316
          PV            284
          RG            313
          RR            161
          SB            518
          SE            154
          SM            538
   

In [71]:
print("GEOGRAPHY PRECINCTS")

print(
    precincts_gdf.loc[
        precincts_gdf["county_key"] == "HENDERSON",
        ["precinct_raw"]
    ]
    .sort_values("precinct_raw")
    .to_string(index=False)
)

GEOGRAPHY PRECINCTS
precinct_raw
          AR
          AT
          BC
          BK
          CB
          CC
          ED
          EF
          ES
          EV
          FL
          FR
          GM
          GR
          HC
          HS
        HV-1
        HV-2
        HV-3
          LJ
          LP
          MG
          NB
          NE
          NM
          NW
          PR
          PV
          RG
          RR
          SB
          SE
          SM
          SW


## Normalize leading zero mismatches with decimal
Normalize only the numeric part before the decimal

In [73]:
def normalize_decimal_precinct(value):
    value = str(value).strip().upper()

    parts = value.split(".")

    if len(parts) == 2:
        left, right = parts

        if left.isdigit() and right.isdigit():
            return f"{int(left)}.{right}"

    return value

In [74]:
normalize_decimal_precinct("01.1")
# "1.1"

normalize_decimal_precinct("078.1")
# "78.1"

normalize_decimal_precinct("08.2")
# "8.2"

'8.2'

In [75]:
results_remaining = unmatched_results.copy()
geo_remaining = no_senate_record.copy()

results_remaining["decimal_key"] = (
    results_remaining["precinct_raw"]
    .apply(normalize_decimal_precinct)
)

geo_remaining["decimal_key"] = (
    geo_remaining["precinct_raw"]
    .apply(normalize_decimal_precinct)
)

In [76]:
decimal_candidates = results_remaining.merge(
    geo_remaining,
    on=["county_key", "decimal_key"],
    how="inner",
    suffixes=("_results", "_geo")
)

In [77]:
decimal_candidates[
    [
        "county_key",
        "precinct_raw_results",
        "precinct_raw_geo"
    ]
].sort_values(
    ["county_key", "precinct_raw_results"]
)

,county_key,precinct_raw_results,precinct_raw_geo
0,BUNCOMBE,1.1,01.1
1,BUNCOMBE,2.1,02.1
2,BUNCOMBE,3.1,03.1
3,BUNCOMBE,4.1,04.1
4,BUNCOMBE,5.1,05.1
5,BUNCOMBE,6.1,06.1
6,BUNCOMBE,7.1,07.1
7,BUNCOMBE,8.2,08.2
8,BUNCOMBE,8.3,08.3
9,BUNCOMBE,9.1,09.1


## Check ambiguity exactly as before
Even though these look unambiguous, keep the safeguard:

In [78]:
decimal_match_counts = (
    decimal_candidates
    .groupby(
        ["county_key", "precinct_raw_results"]
    )
    .size()
    .reset_index(name="match_count")
)

decimal_match_counts[
    decimal_match_counts["match_count"] > 1
]

,county_key,precinct_raw_results,match_count


## Add to the crosswalk

In [79]:
decimal_crosswalk = decimal_candidates[
    [
        "county_key",
        "precinct_raw_results",
        "precinct_raw_geo"
    ]
].copy()

decimal_crosswalk = decimal_crosswalk.rename(
    columns={
        "precinct_raw_results": "results_id",
        "precinct_raw_geo": "geo_id"
    }
)

decimal_crosswalk["match_method"] = "decimal_leading_zero"

In [80]:
crosswalk = pd.concat(
    [
        crosswalk,
        decimal_crosswalk
    ],
    ignore_index=True
)

In [81]:
crosswalk["match_method"].value_counts()

match_method
exact                   2223
leading_zero             398
decimal_leading_zero      11
Name: count, dtype: int64

## Add specific one-offs to crosswalk

In [82]:
cv_record = pd.DataFrame({
    "county_key": ["HENDERSON"],
    "results_id": ["CV"],
    "geo_id": [pd.NA],
    "match_method": ["no_geometry"]
})

In [83]:
crosswalk = pd.concat(
    [
        crosswalk,
        cv_record
    ],
    ignore_index=True
)

In [84]:
crosswalk["match_method"].value_counts()

match_method
exact                   2223
leading_zero             398
decimal_leading_zero      11
no_geometry                1
Name: count, dtype: int64

In [86]:
crosswalk

,county_key,results_id,geo_id,match_method
0,ALAMANCE,03C,03C,exact
1,ALAMANCE,03N,03N,exact
2,ALAMANCE,03N2,03N2,exact
3,ALAMANCE,03SE,03SE,exact
4,ALAMANCE,03SM,03SM,exact
...,...,...,...,...
2628,BUNCOMBE,8.2,08.2,decimal_leading_zero
2629,BUNCOMBE,8.3,08.3,decimal_leading_zero
2630,BUNCOMBE,9.1,09.1,decimal_leading_zero
2631,MECKLENBURG,78.1,078.1,decimal_leading_zero


## Save to CSV

In [85]:
output_file_path = os.path.join(script_dir, '..', 'data', 'crosswalks', 'precinct_crosswalk_2026.csv')

In [87]:
crosswalk.to_csv(output_file_path, index=False)

## Time to merge

In [89]:
senate_xwalk = contest_summary_df.merge(
    crosswalk,
    left_on=["county_key", "precinct_raw"],
    right_on=["county_key", "results_id"],
    how="left",
    validate="one_to_one"
)

In [90]:
senate_xwalk["match_method"].value_counts(dropna=False)

match_method
exact                   2223
leading_zero             398
decimal_leading_zero      11
no_geometry                1
Name: count, dtype: int64

In [91]:
# Now split the mappable results from the unmappable Henderson record

In [92]:
senate_mappable = senate_xwalk[
    senate_xwalk["geo_id"].notna()
].copy()

senate_unmapped = senate_xwalk[
    senate_xwalk["geo_id"].isna()
].copy()

In [94]:
len(senate_mappable)
# 2632

# len(senate_unmapped)
# 1

2632

In [95]:
senate_unmapped[
    ["county_key", "precinct_raw", "contest_votes", "match_method"]
]

,county_key,precinct_raw,contest_votes,match_method
1217,HENDERSON,CV,115,no_geometry


In [99]:
# Join the mappable results to the full statewide precinct geometry

In [97]:
senate_map = precincts_gdf.merge(
    senate_mappable,
    left_on=["county_key", "precinct_raw"],
    right_on=["county_key", "geo_id"],
    how="left",
    validate="one_to_one",
    indicator="contest_merge"
)

In [98]:
senate_map["contest_merge"].value_counts()

contest_merge
both          2632
left_only        0
right_only       0
Name: count, dtype: int64

In [100]:
# Handle the zero-vote precincts. The main rule is: do not let a 0-vote precinct inherit a fake winner from idxmax() or similar logic.

In [101]:
zero_mask = senate_map["contest_votes"].eq(0)

senate_map.loc[zero_mask, "winner"] = pd.NA
senate_map.loc[zero_mask, "winner_votes"] = 0
senate_map.loc[zero_mask, "winner_share"] = pd.NA
senate_map.loc[zero_mask, "runner_up"] = pd.NA
senate_map.loc[zero_mask, "runner_up_votes"] = 0
senate_map.loc[zero_mask, "margin_of_victory"] = pd.NA
# senate_map.loc[zero_mask, "opacity"] = 0

In [104]:
# Create an explicit status field
# Use for no_contest_record in the future with district/local contests

In [102]:
senate_map["contest_status"] = "votes"

senate_map.loc[
    senate_map["contest_votes"].eq(0),
    "contest_status"
] = "zero_votes"

In [108]:
senate_map["contest_status"].value_counts()

contest_status
votes         2616
zero_votes      16
Name: count, dtype: int64

One important point: your full Senate result total still comes from contest_summary, not senate_map, 
because **Henderson CV** is excluded spatially:

In [110]:
official_contest_votes = contest_summary_df["contest_votes"].sum()

mapped_contest_votes = senate_map["contest_votes"].sum()

unmapped_contest_votes = senate_unmapped["contest_votes"].sum()

In [112]:
print(official_contest_votes)
print(mapped_contest_votes)
print(unmapped_contest_votes)

527188
527073
115


In [111]:
official_contest_votes == mapped_contest_votes + unmapped_contest_votes

np.True_

## Export GeoJSON for MapBox

In [122]:
senate_map.columns.tolist()

['id',
 'county_id',
 'prec_id',
 'enr_desc',
 'county_nam',
 'Shape_Leng',
 'Shape_Area',
 'of_prec_id',
 'geometry',
 'county_key',
 'precinct_raw_x',
 'Contest Name',
 'County',
 'Precinct',
 'county_precinct',
 'contest_votes',
 'winner',
 'winner_votes',
 'runner_up',
 'runner_up_votes',
 'winner_vote_share',
 'runner_up_vote_share',
 'margin_of_victory',
 'all_candidate_results',
 'precinct_raw_y',
 'results_id',
 'geo_id',
 'match_method',
 'contest_merge',
 'winner_share',
 'contest_status']

In [121]:
senate_map[["precinct_raw_x", "precinct_raw_y", "prec_id"]].head(20)

,precinct_raw_x,precinct_raw_y,prec_id
0,LI-1,LI-1,LI-1
1,DR,DR,DR
2,RC,RC,RC
3,WS-1,WS-1,WS-1
4,ST,ST,ST
5,RE-1,RE-1,RE-1
6,WI,WI,WI
7,RE-2,RE-2,RE-2
8,MO,MO,MO
9,ED,ED,ED


In [123]:
senate_map = senate_map.rename(
    columns={
        "precinct_raw_x": "geo_precinct_id",
        "precinct_raw_y": "results_precinct_id"
    }
)

In [124]:
senate_map["precinct_id"] = senate_map["geo_precinct_id"]

In [125]:
senate_map.columns.tolist()

['id',
 'county_id',
 'prec_id',
 'enr_desc',
 'county_nam',
 'Shape_Leng',
 'Shape_Area',
 'of_prec_id',
 'geometry',
 'county_key',
 'geo_precinct_id',
 'Contest Name',
 'County',
 'Precinct',
 'county_precinct',
 'contest_votes',
 'winner',
 'winner_votes',
 'runner_up',
 'runner_up_votes',
 'winner_vote_share',
 'runner_up_vote_share',
 'margin_of_victory',
 'all_candidate_results',
 'results_precinct_id',
 'results_id',
 'geo_id',
 'match_method',
 'contest_merge',
 'winner_share',
 'contest_status',
 'precinct_id']

## Create map_key for MapBox

In [130]:
senate_map.rename(columns={"county_precinct": "map_key"}, inplace=True)

## Add tie column

In [149]:
senate_map["is_tie"] = (
    senate_map["contest_votes"].gt(0)
    & senate_map["winner_votes"].eq(senate_map["runner_up_votes"])
)

In [151]:
senate_map["is_tie"].value_counts()

is_tie
False    2604
True       28
Name: count, dtype: int64

## Add participating column for MapBox on-hover and click features

Contest record + 500 votes  → participated = True

Contest record +   0 votes  → participated = True

No contest record           → participated = False

In [131]:
senate_map["participated"] = (
    senate_map["contest_merge"] == "both"
)

In [150]:
senate_map[senate_map["contest_votes"] == 0]

,id,county_id,prec_id,enr_desc,county_nam,Shape_Leng,Shape_Area,of_prec_id,geometry,county_key,...,results_precinct_id,results_id,geo_id,match_method,contest_merge,winner_share,contest_status,precinct_id,participated,is_tie
81,727,32,12,MONUMENT OF FAITH CHURCH,DURHAM,13937.780640,8.738686e+06,NaN,"POLYGON ((2031829.354 810866.354, 2031745.03 8...",DURHAM,...,12,12,12,exact,both,<NA>,zero_votes,12,True,False
416,425,60,056,056,MECKLENBURG,34886.513440,5.778728e+07,NaN,"POLYGON ((1454794.351 551722.749, 1454521.239 ...",MECKLENBURG,...,56,56,056,leading_zero,both,<NA>,zero_votes,056,True,False
641,10058,34,301,ASHLEY ELEMENTARY SCHOOL,FORSYTH,18384.576812,1.345322e+07,NaN,"POLYGON ((1639326.592 862652.184, 1639260.548 ...",FORSYTH,...,301,301,301,exact,both,<NA>,zero_votes,301,True,False
735,764,33,1202,ROCKY MOUNT 2,EDGECOMBE,35734.820155,3.901044e+07,NaN,"POLYGON ((2361629.539 790293.958, 2361623.675 ...",EDGECOMBE,...,1202,1202,1202,exact,both,<NA>,zero_votes,1202,True,False
891,924,89,16,SOUTH FORK,TYRRELL,157630.059285,1.155475e+09,NaN,"POLYGON ((2790521.702 718121.026, 2790493.752 ...",TYRRELL,...,16,16,16,exact,both,<NA>,zero_votes,16,True,False
986,1839,41,G68,G68,GUILFORD,18334.170729,1.811065e+07,NaN,"POLYGON ((1772311.784 846042.497, 1772320.554 ...",GUILFORD,...,G68,G68,G68,exact,both,<NA>,zero_votes,G68,True,False
1201,1356,32,55-49,055-49,DURHAM,11089.961866,6.307735e+06,NaN,"POLYGON ((2029031.522 807855.93, 2029058.104 8...",DURHAM,...,55-49,55-49,55-49,exact,both,<NA>,zero_votes,55-49,True,False
1225,1295,32,41,WHITE ROCK BAPTIST CHURCH,DURHAM,24364.817226,2.437085e+07,NaN,"POLYGON ((2026525.706 801345.661, 2026410.924 ...",DURHAM,...,41,41,41,exact,both,<NA>,zero_votes,41,True,False
1279,1625,82,CLEA,"CLINTON, EAST",SAMPSON,202587.816801,9.450471e+08,NaN,"POLYGON ((2207291.075 449852.649, 2206990.28 4...",SAMPSON,...,CLEA,CLEA,CLEA,exact,both,<NA>,zero_votes,CLEA,True,False
1281,1627,82,CLNE,"CLINTON, NORTHEAST",SAMPSON,65575.256776,7.989404e+07,NaN,"POLYGON ((2205263.638 455478.61, 2205334.426 4...",SAMPSON,...,CLNE,CLNE,CLNE,exact,both,<NA>,zero_votes,CLNE,True,False


In [152]:
senate_web = senate_map[
    [
        "map_key",
        "county_key",
        "precinct_id",
        "results_precinct_id",
        "enr_desc",
        "winner",
        "winner_votes",
        "winner_vote_share",
        "runner_up",
        "runner_up_votes",
        "runner_up_vote_share",
        "contest_votes",
        "all_candidate_results",
        "margin_of_victory",
        "contest_status",
        "participated",
        "is_tie",
        "geometry"
    ]
].copy()

In [153]:
senate_web.crs

<Projected CRS: EPSG:2264>
Name: NAD83 / North Carolina (ftUS)
Axis Info [cartesian]:
- X[east]: Easting (US survey foot)
- Y[north]: Northing (US survey foot)
Area of Use:
- name: United States (USA) - North Carolina - counties of Alamance; Alexander; Alleghany; Anson; Ashe; Avery; Beaufort; Bertie; Bladen; Brunswick; Buncombe; Burke; Cabarrus; Caldwell; Camden; Carteret; Caswell; Catawba; Chatham; Cherokee; Chowan; Clay; Cleveland; Columbus; Craven; Cumberland; Currituck; Dare; Davidson; Davie; Duplin; Durham; Edgecombe; Forsyth; Franklin; Gaston; Gates; Graham; Granville; Greene; Guilford; Halifax; Harnett; Haywood; Henderson; Hertford; Hoke; Hyde; Iredell; Jackson; Johnston; Jones; Lee; Lenoir; Lincoln; Macon; Madison; Martin; McDowell; Mecklenburg; Mitchell; Montgomery; Moore; Nash; New Hanover; Northampton; Onslow; Orange; Pamlico; Pasquotank; Pender; Perquimans; Person; Pitt; Polk; Randolph; Richmond; Robeson; Rockingham; Rowan; Rutherford; Sampson; Scotland; Stanly; Stokes; Sur

In [154]:
senate_web = senate_web.to_crs("EPSG:4326")

In [155]:
contest_name = senate_map['Contest Name'].unique()[0]
contest_name

'US SENATE (REP)'

In [156]:
processed_file_path = os.path.join(script_dir, '..', 'data', 'processed', f'{pydash.snake_case(contest_name)}_contest_summary.geojson')
map_data_file_path = os.path.join(script_dir, '..', 'map', 'data', f'{pydash.snake_case(contest_name)}_contest_summary.geojson')

In [157]:
senate_web.to_file(processed_file_path, driver="GeoJSON")
senate_web.to_file(map_data_file_path, driver="GeoJSON")

## Add bounds to contest.js

In [148]:
participating = senate_web[
    senate_web["participated"]
].copy()

west, south, east, north = participating.total_bounds

contest_bounds = [
    [west, south],
    [east, north]
]

print(contest_bounds)

[[np.float64(-84.3218209914704), np.float64(33.75287799999573)], [np.float64(-75.4001189911457), np.float64(36.588136999894786)]]
